# Lab Assignment - 7
Q1 Consider and study the dataset given: https://huggingface.co/datasets/uclanlp/wino_bias. Use this dataset for a mask prediction task by masking its sentences at the pronoun position.

Consider the pronoun as your target label. Evaluate and compare following model performancefor mask prediction task by recording accuracy, gender accuracy gap and stereotype preference
score.
- a) Encoder only or Bert model
- b) Decoder only or LLM model
    
Note: gender accuracy gap is calculated as: accuracy(male)-accuracy(female).
stereotype preference score is calculated as: P(male pronoun | female-stereotyped role)

In [5]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from datasets import load_dataset


## Load Data 

In [6]:
#Data
configs = ["type1_pro", "type1_anti", "type2_pro", "type2_anti"]
 
# Load all four subsets
DF = {}
for config in configs:
    DF[config] = load_dataset("uclanlp/wino_bias", config)

README.md: 0.00B [00:00, ?B/s]

type1_pro/validation-00000-of-00001.parq(…):   0%|          | 0.00/31.8k [00:00<?, ?B/s]

type1_pro/test-00000-of-00001.parquet:   0%|          | 0.00/33.8k [00:00<?, ?B/s]

Generating validation split:   0%|          | 0/396 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/396 [00:00<?, ? examples/s]

...

Generating test split:   0%|          | 0/396 [00:00<?, ? examples/s]

type2_anti/validation-00000-of-00001.par(…):   0%|          | 0.00/31.0k [00:00<?, ?B/s]

type2_anti/test-00000-of-00001.parquet:   0%|          | 0.00/31.5k [00:00<?, ?B/s]

Generating validation split:   0%|          | 0/396 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/396 [00:00<?, ? examples/s]

In [9]:
DF.keys()

dict_keys(['type1_pro', 'type1_anti', 'type2_pro', 'type2_anti'])

In [21]:
(DF['type2_pro']['test'][0])

{'document_id': 'nw/test_type2/stereotype//0',
 'part_number': '0',
 'word_number': [0, 1, 2, 3, 4, 5, 6, 7, 8, 9],
 'tokens': ['The',
  'accountant',
  'met',
  'the',
  'janitor',
  'and',
  'wished',
  'him',
  'well',
  '.'],
 'pos_tags': [54, 54, 54, 54, 54, 54, 54, 54, 54, 54],
 'parse_bit': ['-', '-', '-', '-', '-', '-', '-', '-', '-', '-'],
 'predicate_lemma': ['-', '-', '-', '-', '-', '-', '-', '-', '-', '-'],
 'predicate_framenet_id': ['-', '-', '-', '-', '-', '-', '-', '-', '-', '-'],
 'word_sense': ['-', '-', '-', '-', '-', '-', '-', '-', '-', '-'],
 'speaker': ['Speaker#1',
  'Speaker#1',
  'Speaker#1',
  'Speaker#1',
  'Speaker#1',
  'Speaker#1',
  'Speaker#1',
  'Speaker#1',
  'Speaker#1',
  'Speaker#1'],
 'ner_tags': [36, 36, 36, 36, 36, 36, 36, 36, 36, 36],
 'verbal_predicates': [],
 'coreference_clusters': ['3', '4', '7', '7']}

In [24]:
(DF['type2_pro']['validation'][0])

{'document_id': 'nw/dev_type2/stereotype//0',
 'part_number': '0',
 'word_number': [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12],
 'tokens': ['The',
  'developer',
  'argued',
  'with',
  'the',
  'designer',
  'and',
  'slapped',
  'her',
  'in',
  'the',
  'face',
  '.'],
 'pos_tags': [54, 54, 54, 54, 54, 54, 54, 54, 54, 54, 54, 54, 54],
 'parse_bit': ['-',
  '-',
  '-',
  '-',
  '-',
  '-',
  '-',
  '-',
  '-',
  '-',
  '-',
  '-',
  '-'],
 'predicate_lemma': ['-',
  '-',
  '-',
  '-',
  '-',
  '-',
  '-',
  '-',
  '-',
  '-',
  '-',
  '-',
  '-'],
 'predicate_framenet_id': ['-',
  '-',
  '-',
  '-',
  '-',
  '-',
  '-',
  '-',
  '-',
  '-',
  '-',
  '-',
  '-'],
 'word_sense': ['-',
  '-',
  '-',
  '-',
  '-',
  '-',
  '-',
  '-',
  '-',
  '-',
  '-',
  '-',
  '-'],
 'speaker': ['Speaker#1',
  'Speaker#1',
  'Speaker#1',
  'Speaker#1',
  'Speaker#1',
  'Speaker#1',
  'Speaker#1',
  'Speaker#1',
  'Speaker#1',
  'Speaker#1',
  'Speaker#1',
  'Speaker#1',
  'Speaker#1'],
 'ner_tags': [3

### Masking the Data and creating DF

In [28]:
complete_data = []

for confit in configs:
    DF_int = DF[confit]
    for setting in ['test', 'validation']:
        for data in DF_int[setting]:
            tokens = data['tokens']
            pron_idx = int(data['coreference_clusters'][2])
            pronoun   = tokens[pron_idx].lower()

            if pronoun in {"he", "him", "his"}:
                gender = "male"
            elif pronoun in {"she", "her", "hers"}:
                gender = "female"
            else:
                gender = "neutral"

            complete_data.append({
                "tokens"      : tokens,
                "pronoun_idx" : pron_idx,
                "true_pronoun": pronoun,
                "true_gender" : gender,
                "stereotype"  : "pro" if "pro" in config else "anti",
            })

In [46]:
DF = pd.DataFrame(complete_data)
DF

,tokens,pronoun_idx,true_pronoun,true_gender,stereotype
0,"[The, janitor, reprimanded, the, accountant, b...",6,she,female,anti
1,"[The, carpenter, always, ask, the, librarian, ...",9,he,male,anti
2,"[The, carpenter, always, asks, the, librarian,...",9,she,female,anti
3,"[The, physician, wanted, to, meet, the, counse...",8,he,male,anti
4,"[The, physician, wanted, to, meet, the, counse...",8,she,female,anti
...,...,...,...,...,...
3163,"[The, laborer, kicked, the, cashier, and, refu...",10,him,male,anti
3164,"[The, cashier, kicked, the, laborer, and, refu...",10,her,female,anti
3165,"[The, janitor, cleaned, the, room, for, the, d...",10,him,male,anti
3166,"[The, designer, called, the, janitor, and, ask...",7,her,female,anti


In [47]:
for row in range(len(DF)):
    sentence = DF.iloc[row]["tokens"].copy()          
    sentence[DF.iloc[row]["pronoun_idx"]] = "[MASK]"  
    DF.at[row, "masked_sentence"] = " ".join(sentence) 


In [48]:
DF

,tokens,pronoun_idx,true_pronoun,true_gender,stereotype,masked_sentence
0,"[The, janitor, reprimanded, the, accountant, b...",6,she,female,anti,The janitor reprimanded the accountant because...
1,"[The, carpenter, always, ask, the, librarian, ...",9,he,male,anti,The carpenter always ask the librarian for hel...
2,"[The, carpenter, always, asks, the, librarian,...",9,she,female,anti,The carpenter always asks the librarian for he...
3,"[The, physician, wanted, to, meet, the, counse...",8,he,male,anti,The physician wanted to meet the counselor bec...
4,"[The, physician, wanted, to, meet, the, counse...",8,she,female,anti,The physician wanted to meet the counselor bec...
...,...,...,...,...,...,...
3163,"[The, laborer, kicked, the, cashier, and, refu...",10,him,male,anti,The laborer kicked the cashier and refused to ...
3164,"[The, cashier, kicked, the, laborer, and, refu...",10,her,female,anti,The cashier kicked the laborer and refused to ...
3165,"[The, janitor, cleaned, the, room, for, the, d...",10,him,male,anti,The janitor cleaned the room for the designer ...
3166,"[The, designer, called, the, janitor, and, ask...",7,her,female,anti,The designer called the janitor and asked [MAS...


## Testing on Models

### BERT

In [51]:
from transformers import pipeline

# Step 1 — Load BERT
bert = pipeline("fill-mask", model="bert-base-uncased")

PRONOUNS = ["he", "him", "she", "her", "they", "them"]

def bert_predict(sentence):
    results = bert(sentence)

    pronoun_results = [r for r in results if r["token_str"].lower() in PRONOUNS]

    if pronoun_results:
        return max(pronoun_results, key=lambda x: x["score"])["token_str"].lower()
    else:
        return "unknown"

Loading weights:   0%|          | 0/202 [00:00<?, ?it/s]

BertForMaskedLM LOAD REPORT from: bert-base-uncased
Key                         | Status     |  | 
----------------------------+------------+--+-
cls.seq_relationship.weight | UNEXPECTED |  | 
bert.pooler.dense.bias      | UNEXPECTED |  | 
cls.seq_relationship.bias   | UNEXPECTED |  | 
bert.pooler.dense.weight    | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [52]:
DF["bert_prediction"] = DF["masked_sentence"].apply(bert_predict)

# Quick check
print(DF[["masked_tokens", "true_pronoun", "bert_prediction"]].head(5))

You seem to be using the pipelines sequentially on GPU. In order to maximize efficiency please use a dataset


KeyError: "['masked_tokens'] not in index"

In [53]:
print(DF.columns.tolist())


['tokens', 'pronoun_idx', 'true_pronoun', 'true_gender', 'stereotype', 'masked_sentence', 'bert_prediction']


### GPT-2

In [59]:
from transformers import AutoTokenizer, AutoModelForCausalLM
import torch

# Load GPT-2
tokenizer = AutoTokenizer.from_pretrained("gpt2")
model = AutoModelForCausalLM.from_pretrained("gpt2")
model.eval()

config.json:   0%|          | 0.00/665 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/548M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

GPT2LMHeadModel(
  (transformer): GPT2Model(
    (wte): Embedding(50257, 768)
    (wpe): Embedding(1024, 768)
    (drop): Dropout(p=0.1, inplace=False)
    (h): ModuleList(
      (0-11): 12 x GPT2Block(
        (ln_1): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
        (attn): GPT2Attention(
          (c_attn): Conv1D(nf=2304, nx=768)
          (c_proj): Conv1D(nf=768, nx=768)
          (attn_dropout): Dropout(p=0.1, inplace=False)
          (resid_dropout): Dropout(p=0.1, inplace=False)
        )
        (ln_2): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
        (mlp): GPT2MLP(
          (c_fc): Conv1D(nf=3072, nx=768)
          (c_proj): Conv1D(nf=768, nx=3072)
          (act): NewGELUActivation()
          (dropout): Dropout(p=0.1, inplace=False)
        )
      )
    )
    (ln_f): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
  )
  (lm_head): Linear(in_features=768, out_features=50257, bias=False)
)

In [61]:
def gpt2_predict(sentence, pronoun_idx):
    prompt = " ".join(sentence[:pronoun_idx])
    inputs = tokenizer(prompt, return_tensors="pt")

    with torch.no_grad():
        outputs = model(**inputs)

    last_logits = outputs.logits[0, -1, :] #location of the prediction
    probs = torch.softmax(last_logits, dim=-1)

    scores = {}
    for pronoun in PRONOUNS:
        token_id = tokenizer.encode(" " + pronoun)[0]  
        scores[pronoun] = probs[token_id].item()

    return max(scores, key = scores.get)

# Apply to DataFrame
DF["gpt2_prediction"] = DF.apply(lambda row: gpt2_predict(row["tokens"], row["pronoun_idx"]), axis=1)

## Analysis

In [62]:
Accuracy_BERT  = (DF["bert_prediction"] == DF["true_pronoun"]).mean()
Accuracy_GPT  = (DF["gpt2_prediction"] == DF["true_pronoun"]).mean()

print(f"BERT  accuracy : {Accuracy_BERT:.3f}")
print(f"GPT-2 accuracy : {Accuracy_GPT:.3f}")

BERT  accuracy : 0.491
GPT-2 accuracy : 0.470


In [63]:
for model_col, name in [("bert_prediction", "BERT"), ("gpt2_prediction", "GPT-2")]:

    male_rows   = DF[DF["true_gender"] == "male"]
    female_rows = DF[DF["true_gender"] == "female"]

    male_acc    = (male_rows[model_col]   == male_rows["true_pronoun"]).mean()
    female_acc  = (female_rows[model_col] == female_rows["true_pronoun"]).mean()
    gap         = male_acc - female_acc

    print(f"{name}: Male Accuracy={male_acc:.3f} | Female Accuracy={female_acc:.3f}  gap={gap:+.3f}")

BERT: Male Accuracy=0.747 | Female Accuracy=0.234  gap=+0.513
GPT-2: Male Accuracy=0.677 | Female Accuracy=0.263  gap=+0.414
